In [ ]:
import numpy as np
from scipy.linalg import solve
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt


def squareExp_kern(x, y, param):
    sigma, theta = param
    d = np.subtract.outer(x, y)  # d = x_i - x_j
    kern = (sigma ** 2) * np.exp(-theta * d ** 2)
    kernd = 2 * theta * d * kern  # first derivative / x
    kerndd = 2 * theta * kern * (1 - 2 * theta * d**2)  # second derivative
    return {'k': kern, 'kdt': kernd, 'kdsdt': kerndd}

param = np.array([1, 1/0.2**2])  # covariance parameters

# 1. Simulation of sample paths
n = 250
x = np.linspace(0, 1, n)

# Computation of the covariance matrix k(x_i, x_j)
kern = squareExp_kern  
kxx = kern(x, x, param)['k']

# Image plot 
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(kxx, cmap='viridis', origin='lower', extent=[0, 1, 0, 1])
plt.title("kernel plot")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar()

# Simulations
samples = np.random.multivariate_normal(mean=np.zeros(n), cov=kxx, size=5).T
plt.subplot(1, 2, 2)
plt.plot(x, samples)
plt.title("sample paths")
plt.show()


Q: Explain why the choice of theta corresponds to a lengthscale parameter equal to 0.2.

Q: What can you say of the sample paths of the GP associated to a squared exponential kernel?

Q: Explain why we used ['k'] in the code. What contains kern(x, x, param)?

## Gaussian process regression

Let's use GP regression to interpolate a function

In [ ]:
# --- Gaussian process regression ---
# Function, learning set or 'design of experiments'  
def fun(x):
    return x + np.sin(4 * np.pi * x)

X = np.linspace(0.1, 1, 6)
Y = fun(X)

# Plot
plt.figure(figsize=(8, 6))
x_plot = np.linspace(0, 1, n)
plt.plot(x_plot, fun(x_plot), linestyle='dotted', label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()


In [ ]:
# --- Conditional mean and kernel ---
def condMean(x, X, Y, kern, param, nugget=0):
    K = kern(X, X, param)['k']
    Kinv = solve(K + nugget * np.eye(len(K)), np.eye(len(K)))
    kxX = kern(x, X, param)['k']
    return kxX @ Kinv @ Y

def condCov(x, X, kern, param, nugget=0):
    K = kern(X, X, param)['k']
    Kinv = solve(K + nugget * np.eye(len(K)), np.eye(len(K)))
    kxX = kern(x, X, param)['k']
    kxx = kern(x, x, param)['k']
    return kxx - kxX @ Kinv @ kxX.T

param = np.array([1, 1/0.2**2])

mu = condMean(x_plot, X, Y, kern, param, nugget=0)
Sigma = condCov(x_plot, X, kern, param, nugget=0)
varSigma = np.diag(Sigma)
varSigma = np.maximum(varSigma, 0)  # to avoid numerical negative values

plt.figure(figsize=(8, 6))
plt.plot(x_plot, fun(x_plot), linestyle='dotted', label='function f(x)')
plt.scatter(X, Y, color='black', s=100, label='training points')
plt.plot(x_plot, mu, color='blue', linewidth=3, label='cond. mean m(x)')
plt.plot(x_plot, mu + 1.96 * np.sqrt(varSigma), linestyle="dashed", color = "blue", label='95% pred. inter')
plt.plot(x_plot, mu - 1.96 * np.sqrt(varSigma), linestyle="dashed", color = "blue")
plt.legend()
plt.ylim(-1.5, 4)
plt.xlabel("x")
plt.ylabel("y")
plt.show()


## Gaussian process regression with derivatives

In [ ]:
def fund(x):
    omega = 4 * np.pi
    return 1 + omega * np.cos(omega * x)

# Derivative values
Yd = fund(X)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(x, fun(x), label='function f(x)')
plt.scatter(X, Y, color='black', label='training points')
h = 0.05
for i in range(len(X)):
    plt.arrow(X[i], Y[i], h, h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)
    plt.arrow(X[i], Y[i], -h, -h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)

plt.xlabel('x')
plt.ylabel('y')
plt.xlim(0, 1)
plt.ylim(-2, 2)
plt.legend()

plt.show()



In [ ]:
# Conditional mean knowing function & derivative values
def condMeanDer(x, X, Y, Yder, kern, param, nugget=0):
    kXXlist = kern(X, X, param)
    K = np.block([[kXXlist['k'], kXXlist['kdt']],
                  [kXXlist['kdt'].T, kXXlist['kdsdt']]])
    Kinv = np.linalg.inv(K + nugget * np.eye(K.shape[0]))
    kxXlist = kern(x, X, param)
    kxX = np.column_stack((kxXlist['k'], kxXlist['kdt']))
    Y_Yder = np.concatenate((Y, Yder))
    return kxX @ Kinv @ Y_Yder

# Conditional covariance knowing function & derivative values
def condCovDer(x, X, kern, param, nugget=0):
    kXXlist = kern(X, X, param)
    K = np.block([[kXXlist['k'], kXXlist['kdt']],
                  [kXXlist['kdt'].T, kXXlist['kdsdt']]])
    Kinv = np.linalg.inv(K + nugget * np.eye(K.shape[0]))
    kxXlist = kern(x, X, param)
    kxX = np.column_stack((kxXlist['k'], kxXlist['kdt']))
    kxx = kern(x, x, param)['k']
    return kxx - kxX @ Kinv @ kxX.T

Q : How are modified the Kriging equations when using derivatives? Interpret the four blocks of K and the two blocks of KxX.

In [ ]:
# Computation on our data
mcDer = condMeanDer(x, X, Y, Yd, kern, param, nugget=1e-6)
kcDer = condCovDer(x, X, kern, param, nugget=1e-6)
varSigmaDer = np.diag(kcDer)
varSigmaDer = np.maximum(varSigmaDer, 0)  # Éviter les valeurs numériques négatives

# Figure 1 : Cokriging mean and prediction interval
plt.figure(figsize=(10, 6))
plt.plot(x, fun(x), label='function f(x)')
plt.scatter(X, Y, color='black', label='training points')
h = 0.05
for i in range(len(X)):
    plt.arrow(X[i], Y[i], h, h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)
    plt.arrow(X[i], Y[i], -h, -h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)

ubDer = mcDer + 1.96 * np.sqrt(varSigmaDer)
lbDer = mcDer - 1.96 * np.sqrt(varSigmaDer)
plt.plot(x, mcDer, color='blue', label='cond. mean m(x)')

plt.xlabel('x')
plt.ylabel('y')
plt.title("GP regression with derivatives: mean & prediction interval")
plt.legend()
plt.xlim(0, 1)
plt.ylim(-2, 2)
plt.show()


In [ ]:
# Conditional sample-path (with derivatives)
np.random.seed(0)
condSamples = multivariate_normal.rvs(mean=mcDer, cov=kcDer, size=100)  # 100 samples of length len(x)

# Figure 2: Conditional simulations
plt.figure(figsize=(10, 6))

plt.plot(x, fun(x), label='function f(x)', color='black')
plt.scatter(X, Y, color='black', label='training points')
h = 0.05
for i in range(len(X)):
    plt.arrow(X[i], Y[i], h, h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)
    plt.arrow(X[i], Y[i], -h, -h * Yd[i], head_width=0.01, head_length=0.01, fc='black', ec='black', length_includes_head=True)

for sample in condSamples:
    plt.plot(x, sample, linestyle='-', color='gray', alpha=0.1)  # alpha: linewidth

plt.xlabel('x')
plt.ylabel('y')
plt.title("GP regression with derivatives: conditional simulations")
plt.legend()
plt.xlim(0, 1)
plt.ylim(-2, 2)
plt.show()

Q: What is the benefit of using the information of derivatives?